# Sprint 3.2 — Simulator Validation Notebook

**Version History**
| Date | Change | Author |
|------|--------|--------|
| Apr 9, 2026 | Initial creation — 3 validation gates + scenario demos | Claude + Luna |

**Purpose:** Interactive notebook to load, test, and explore `simulator.py`. Runs the same 3 validation gates as `test_simulator.py` but with visible outputs, then demonstrates scenario runs so you can inspect per-tract results.

**How to run:** Open in VS Code, select your Python kernel, and Run All. Requires `xgboost`, `joblib`, `scikit-learn`, `pandas`, `numpy` installed.

## 1. Load Simulator

The `Simulator` class loads three files at init:
- `Sprint3_Baseline_State.csv` — 504 tracts with 13 features + equity context
- `Sprint2b_XGBoost_v3.pkl` — trained XGBoost model (unscaled features)
- `Sprint3_Lever_Catalog.json` — 4 transit-service levers with valid ranges

It also re-predicts the baseline deficit from scratch using the model. This is intentional — the stored `deficit_predicted` values came from a 507-tract training context (the notebook merged 3 duplicate tracts from the equity CSV). Re-predicting on our 504 tracts ensures the proportional-change ratio is perfectly self-consistent. See Lesson 20 in `tasks/lessons.md`.

In [3]:
%pip install xgboost joblib scikit-learn pandas numpy pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytest]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import pandas as pd
from simulator import Simulator

sim = Simulator()
print(f"Tracts loaded:  {sim.n_tracts}")
print(f"Available levers: {sim.lever_names}")
print(f"\nBaseline snapshot (first 5 tracts):")
sim.baseline_df[['tract_geoid', 'equity_priority_score', 'equity_tier',
                  'composite_need', 'composite_access_deficit']].head()

Tracts loaded:  504
Available levers: ['freq_peak_am_tph', 'freq_early_tph', 'weekend_weekday_ratio', 'rail_trip_share']

Baseline snapshot (first 5 tracts):


,tract_geoid,equity_priority_score,equity_tier,composite_need,composite_access_deficit
0,12086000107,0.118780,Moderate,0.213208,0.557106
1,12086000109,0.105278,Moderate,0.184907,0.569358
2,12086000141,0.070206,Low,0.135234,0.519143
3,12086000115,0.070244,Low,0.144310,0.486756
4,12086000118,0.057310,Low,0.125272,0.457487


## 2. Gate 1 — Zero-Delta Test

The most critical validation: running the simulator with **no changes** should produce exactly zero equity drift. This proves the proportional-change pipeline is self-consistent — `deficit_scenario / deficit_baseline = 1.0` exactly, so `equity_after = equity_before`.

**Pass criteria:** max |equity_delta| < 1e-8, zero tier shifts.

In [5]:
r0 = sim.run(access_deltas={}, label="Zero-delta")

max_eq_drift  = np.abs(r0.tract_df["equity_delta"]).max()
max_def_drift = np.abs(r0.tract_df["deficit_delta"]).max()
tier_shifts   = (r0.tract_df["tier_before"] != r0.tract_df["tier_after"]).sum()

print("── Gate 1: Zero-Delta ──")
print(f"  Max |equity_delta|:  {max_eq_drift:.2e}  {'PASS' if max_eq_drift < 1e-8 else 'FAIL'}")
print(f"  Max |deficit_delta|: {max_def_drift:.2e}  {'PASS' if max_def_drift < 1e-8 else 'FAIL'}")
print(f"  Tier shifts:         {tier_shifts}         {'PASS' if tier_shifts == 0 else 'FAIL'}")
print(f"  Summary improved:    {r0.summary['n_improved']}  |  worsened: {r0.summary['n_worsened']}")

── Gate 1: Zero-Delta ──
  Max |equity_delta|:  9.71e-17  PASS
  Max |deficit_delta|: 0.00e+00  PASS
  Tier shifts:         0         PASS
  Summary improved:    0  |  worsened: 0


## 3. Gate 2 — Isolation Test

Apply +2 trips/hr peak AM frequency **only to Critical tracts**. Unaffected tracts (Low, Moderate, High) must have exactly zero equity and deficit change. This validates that the tract filter works correctly and that headway derivation doesn't leak to unfiltered tracts (Lesson 19).

**Pass criteria:** max |equity_delta| < 1e-8 for unaffected tracts; affected tracts show nonzero change.

In [7]:
baseline = sim.baseline_df
critical_tracts = baseline.loc[baseline["equity_tier"] == "Critical", "tract_geoid"].tolist()

r_iso = sim.run(
    access_deltas={"freq_peak_am_tph": 2.0},
    tract_filter=critical_tracts,
    label="Isolation: +2 tph Critical only",
)

unaffected = r_iso.tract_df[~r_iso.tract_df["tract_geoid"].isin(critical_tracts)]
affected   = r_iso.tract_df[ r_iso.tract_df["tract_geoid"].isin(critical_tracts)]

max_unaffected_eq  = np.abs(unaffected["equity_delta"]).max()
max_unaffected_def = np.abs(unaffected["deficit_delta"]).max()
n_affected_changed = (np.abs(affected["deficit_delta"]) > 1e-8).sum()

print("── Gate 2: Isolation ──")
print(f"  Critical tracts targeted:    {len(critical_tracts)}")
print(f"  Unaffected tracts:           {len(unaffected)}")
print(f"  Max |equity_delta| unaffected:  {max_unaffected_eq:.2e}  {'PASS' if max_unaffected_eq < 1e-8 else 'FAIL'}")
print(f"  Max |deficit_delta| unaffected: {max_unaffected_def:.2e}  {'PASS' if max_unaffected_def < 1e-8 else 'FAIL'}")
print(f"  Affected tracts that changed:   {n_affected_changed}/{len(affected)}  {'PASS' if n_affected_changed > 0 else 'FAIL'}")

── Gate 2: Isolation ──
  Critical tracts targeted:    51
  Unaffected tracts:           453
  Max |equity_delta| unaffected:  9.71e-17  PASS
  Max |deficit_delta| unaffected: 0.00e+00  PASS
  Affected tracts that changed:   51/51  PASS


## 4. Gate 3 — Direction Tests

Each lever increase should reduce average deficit, consistent with SHAP rankings from Sprint 2b.5:
- `freq_peak_am_tph` +2 tph (SHAP #1, 39.6%)
- `weekend_weekday_ratio` +0.1 (SHAP #2, 21.7%)
- `freq_early_tph` +1 tph (SHAP #3, 6.9%)
- `rail_trip_share` +0.10 (largest single lever in what-if: -0.0149 deficit)

We also check SHAP ranking consistency: S1 (peak freq) should produce a bigger deficit reduction than S3 (early freq), matching SHAP #1 > SHAP #3.

**Pass criteria:** all avg_deficit_delta < 0; |S1 delta| > |S3 delta|.

In [8]:
scenarios = [
    ("S1: Peak freq +2 tph",      {"freq_peak_am_tph": 2.0}),
    ("S2: Weekend ratio +0.1",    {"weekend_weekday_ratio": 0.1}),
    ("S3: Early freq +1 tph",     {"freq_early_tph": 1.0}),
    ("S4: Rail share +10pp",      {"rail_trip_share": 0.10}),
]

print("── Gate 3: Direction Tests ──\n")
print(f"{'Scenario':<28} {'Avg Deficit Δ':>14} {'Tracts Improved':>16} {'Tier Upgrades':>14} {'Result':>8}")
print("─" * 84)

results = {}
for label, deltas in scenarios:
    r = sim.run(access_deltas=deltas, label=label)
    results[label] = r
    s = r.summary
    passed = s["avg_deficit_delta"] < 0
    print(f"{label:<28} {s['avg_deficit_delta']:>+14.6f} {s['n_improved']:>12}/504 {s['n_tier_upgrades']:>14} {'PASS' if passed else 'FAIL':>8}")

# SHAP ranking consistency: |S1| > |S3|
s1_delta = results["S1: Peak freq +2 tph"].summary["avg_deficit_delta"]
s3_delta = results["S3: Early freq +1 tph"].summary["avg_deficit_delta"]
shap_ok = abs(s1_delta) > abs(s3_delta)
print(f"\nSHAP ranking: |S1|={abs(s1_delta):.6f} > |S3|={abs(s3_delta):.6f}  {'PASS' if shap_ok else 'FAIL'}")

── Gate 3: Direction Tests ──

Scenario                      Avg Deficit Δ  Tracts Improved  Tier Upgrades   Result
────────────────────────────────────────────────────────────────────────────────────
S1: Peak freq +2 tph              -0.026223          494/504             74     PASS
S2: Weekend ratio +0.1            -0.007428          412/504             23     PASS
S3: Early freq +1 tph             -0.003855          309/504             14     PASS
S4: Rail share +10pp              -0.016898          487/504             49     PASS

SHAP ranking: |S1|=0.026223 > |S3|=0.003855  PASS


## 5. Tier Shift Matrix

The 4×4 tier shift matrix shows how many tracts move between tiers for each scenario. Rows are the original tier, columns are the new tier. Diagonal = tracts that stayed in the same tier.

This is the core output Deloitte cares about: "if we add 2 trips/hr in peak AM, how many Critical tracts drop to High?"

In [9]:
for label, _ in scenarios:
    r = results[label]
    print(f"\n{'═'*50}")
    print(f"  {label}")
    print(f"  Upgrades: {r.summary['n_tier_upgrades']}  |  Downgrades: {r.summary['n_tier_downgrades']}")
    print(f"{'═'*50}")
    print(r.tier_shifts)
    print()


══════════════════════════════════════════════════
  S1: Peak freq +2 tph
  Upgrades: 74  |  Downgrades: 0
══════════════════════════════════════════════════
tier_after   Low  Moderate  High  Critical
tier_before                               
Low          200         0     0         0
Moderate      27       123     0         0
High           0        31    71         0
Critical       0         0    16        36


══════════════════════════════════════════════════
  S2: Weekend ratio +0.1
  Upgrades: 23  |  Downgrades: 7
══════════════════════════════════════════════════
tier_after   Low  Moderate  High  Critical
tier_before                               
Low          195         5     0         0
Moderate      10       138     2         0
High           0         9    93         0
Critical       0         0     4        48


══════════════════════════════════════════════════
  S3: Early freq +1 tph
  Upgrades: 14  |  Downgrades: 2
══════════════════════════════════════════════════
ti

## 6. Per-Tract Detail — S4 Rail Modal Shift

S4 is the most impactful single-lever scenario (Sprint 2b.5 what-if: +10pp rail share → -0.0149 deficit, 98% tracts improve). Let's inspect the top-improving and top-worsening tracts, plus fragile tract impacts.

In [10]:
r4 = results["S4: Rail share +10pp"]
df4 = r4.tract_df.sort_values("equity_delta")

print("── Top 10 most improved tracts (largest equity decrease) ──")
print(df4[["tract_geoid", "equity_before", "equity_after", "equity_delta",
           "tier_before", "tier_after"]].head(10).to_string(index=False))

print("\n── Top 5 most worsened tracts (if any) ──")
worsened = df4[df4["equity_delta"] > 0]
if len(worsened) > 0:
    print(worsened[["tract_geoid", "equity_before", "equity_after", "equity_delta",
                     "tier_before", "tier_after"]].tail(5).to_string(index=False))
else:
    print("  No tracts worsened.")

print(f"\n── Fragile tracts impact ──")
fragile = r4.tract_df[r4.tract_df["is_fragile"] == True]
if len(fragile) > 0:
    print(f"  Fragile tracts: {len(fragile)}")
    print(f"  Fragile improved: {(fragile['equity_delta'] < -1e-8).sum()}")
    print(f"  Fragile avg equity Δ: {fragile['equity_delta'].mean():.6f}")
else:
    print("  No fragile tracts flagged.")

── Top 10 most improved tracts (largest equity decrease) ──
 tract_geoid  equity_before  equity_after  equity_delta tier_before tier_after
 12086008904       0.452849      0.405224     -0.047624    Critical   Critical
 12086001501       0.276751      0.243910     -0.032841    Critical   Critical
 12086002003       0.225858      0.200230     -0.025627    Critical   Critical
 12086001901       0.203088      0.177935     -0.025154    Critical   Critical
 12086001401       0.215299      0.190344     -0.024955    Critical   Critical
 12086001005       0.231516      0.207181     -0.024335    Critical   Critical
 12086002001       0.204610      0.181825     -0.022785    Critical   Critical
 12086000408       0.189853      0.167641     -0.022212    Critical   Critical
 12086001903       0.183896      0.162605     -0.021291    Critical   Critical
 12086002300       0.169990      0.148883     -0.021107    Critical       High

── Top 5 most worsened tracts (if any) ──
 tract_geoid  equity_before 

## 7. Run `test_simulator.py` (pytest)

This cell runs the full 16-test suite from `test_simulator.py` directly. It's the same thing as running `pytest test_simulator.py -v` from the terminal but inside the notebook so you can see results here. Same as Gate 1-3 tests just all at once.

The 16 tests cover:
- **Zero-delta (4):** equity unchanged, deficit unchanged, no tier shifts, summary all zeros
- **Isolation (2):** unaffected tracts untouched, affected tracts changed
- **Direction (4):** all 4 levers reduce deficit when increased
- **Structural (6):** invalid lever error, 504 tracts, shapes, equity bounds, SHAP consistency

In [11]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_simulator.py", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

============================= test session starts ==============================
collecting ... collected 16 items

test_simulator.py::TestZeroDelta::test_equity_unchanged PASSED           [  6%]
test_simulator.py::TestZeroDelta::test_deficit_unchanged PASSED          [ 12%]
test_simulator.py::TestZeroDelta::test_no_tier_shifts PASSED             [ 18%]
test_simulator.py::TestZeroDelta::test_summary_zeros PASSED              [ 25%]
test_simulator.py::TestIsolation::test_unaffected_tracts_unchanged PASSED [ 31%]
test_simulator.py::TestIsolation::test_affected_tracts_did_change PASSED [ 37%]
test_simulator.py::TestDirection::test_freq_peak_am_decreases_deficit PASSED [ 43%]
test_simulator.py::TestDirection::test_weekend_ratio_decreases_deficit PASSED [ 50%]
test_simulator.py::TestDirection::test_rail_share_decreases_deficit PASSED [ 56%]
test_simulator.py::TestDirection::test_freq_early_decreases_deficit PASSED [ 62%]
test_simulator.py::TestStructural::test_invalid_lever_raises PASSED   

## 8. Combined Multi-Lever Scenario

Quick demo of combining levers — this is what Sprint 3.3 S5 will look like. The simulator accepts multiple deltas in a single `run()` call. Here we combine peak freq boost + rail share increase.

In [12]:
r_combined = sim.run(
    access_deltas={
        "freq_peak_am_tph": 2.0,
        "rail_trip_share": 0.10,
        "weekend_weekday_ratio": 0.1,
    },
    label="S5: Combined (peak +2, rail +10pp, weekend +0.1)",
)
sc = r_combined.summary

print(f"── {r_combined.label} ──")
print(f"  Avg deficit Δ:     {sc['avg_deficit_delta']:+.6f}")
print(f"  Avg equity Δ:      {sc['avg_equity_delta']:+.6f}")
print(f"  Tracts improved:   {sc['n_improved']}/504")
print(f"  Tier upgrades:     {sc['n_tier_upgrades']}")
print(f"  Tier downgrades:   {sc['n_tier_downgrades']}")
print(f"  Max improvement:   {sc['max_equity_improvement']:+.6f}")

# Verify combined > any single lever
s1_d = abs(results["S1: Peak freq +2 tph"].summary["avg_deficit_delta"])
s4_d = abs(results["S4: Rail share +10pp"].summary["avg_deficit_delta"])
combined_d = abs(sc["avg_deficit_delta"])
print(f"\n  Combined ({combined_d:.6f}) > S1 alone ({s1_d:.6f})? {combined_d > s1_d}")
print(f"  Combined ({combined_d:.6f}) > S4 alone ({s4_d:.6f})? {combined_d > s4_d}")

── S5: Combined (peak +2, rail +10pp, weekend +0.1) ──
  Avg deficit Δ:     -0.051962
  Avg equity Δ:      -0.012528
  Tracts improved:   502/504
  Tier upgrades:     148
  Tier downgrades:   0
  Max improvement:   -0.085593

  Combined (0.051962) > S1 alone (0.026223)? True
  Combined (0.051962) > S4 alone (0.016898)? True
